# Step 3: LoRA Fine-tuning
Use **PEFT LoRA** to fine-tune **HuggingFaceTB/SmolLM2-1.7B-Instruct** for the NBA player prop betting analysis task.
**Workflow**:
1. Load the base model and tokenizer
2. Read `train.jsonl` and convert to SmolLM2 Chat Template format
3. Configure LoraConfig (r=8, target_modules: q_proj, v_proj)
4. Wrap with `get_peft_model()`
5. Train using `transformers.Trainer` (DataCollatorForSeq2Seq)
6. Save to `outputs/smollm2-bet-advisor`
**Before running**: Requires GPU (recommended) or CPU for small batch testing. Run from the project root directory.

## 1. Path Configuration

In [1]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "finetune":
    project_root = project_root.parent.parent
elif project_root.name == "scripts":
    project_root = project_root.parent

data_dir = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "train.jsonl"
adapter_path = output_dir / "smollm2-bet-advisor"

print(f"Project root: {project_root}")
print(f"Train: {train_path} (exists: {train_path.exists()})")
print(f"Adapter output: {adapter_path}")

Project root: /Users/wuyusen/Documents/bet
Train: /Users/wuyusen/Documents/bet/data/train.jsonl (exists: True)
Adapter output: /Users/wuyusen/Documents/bet/outputs/smollm2-bet-advisor


## 2. Load Base Model and Tokenizer

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

print(f"Model loaded. Device: {next(model.parameters()).device}")

/Users/wuyusen/Documents/bet/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer: HuggingFaceTB/SmolLM2-1.7B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: HuggingFaceTB/SmolLM2-1.7B-Instruct


Loading weights: 100%|██████████| 218/218 [00:06<00:00, 32.60it/s, Materializing param=model.norm.weight]                              


Model loaded. Device: cpu


## 3. System Prompt & Load Training Data

Use the same SYSTEM_PROMPT as in base_inference to ensure consistency between training and inference formats.

In [ ]:
import json

SYSTEM_PROMPT = """You are an expert NBA player data analyst. Your task is to analyze betting questions (over/under) using historical player statistics.

## Tree of Thought Reasoning Framework

Build your reasoning as a tree with these main branches (evaluate each, then synthesize):

1. **Sample & Statistics Branch**: For each context filter (all games, last N games, starter vs bench, with/without star teammates), assess:
   - n_games: Is sample size sufficient? (n < 10 → low weight)
   - p_over, p_under, mean, std: Which context favors over vs under?
   - Conflict between contexts → note uncertainty

2. **Lineup/Teammate Branch**: How does starter vs bench, or with/without star teammates, change the stats? Does lineup context support or contradict the main trend?

3. **Risk & Synthesis Branch**: Given the above, what is the net signal? If branches conflict or sample is weak → prefer "avoid".

## Output Format (JSON only, no markdown)

{
  "decision": "over" | "under" | "avoid",
  "confidence": 0.0 to 1.0,
  "reasoning": {
    "tree_of_thought": [
      {"step": 1, "branch": "sample_stats", "thought": "...", "conclusion": "..."},
      {"step": 2, "branch": "lineup_teammate", "thought": "...", "conclusion": "..."},
      {"step": 3, "branch": "synthesis", "thought": "...", "conclusion": "..."}
    ]
  },
  "summary": "One-sentence conclusion"
}

Each step must have: branch (which dimension), thought (your analysis), conclusion (what this branch implies for the decision).
Respond with ONLY valid JSON, no markdown or extra text."""

def load_jsonl(path: Path) -> list[dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

train_data = load_jsonl(train_path)
print(f"Train: {len(train_data)} records")
if train_data:
    ex = train_data[0]
    print(f"  instruction: {ex['instruction'][:60]}...")
    print(f"  input: {len(ex['input'])} chars, output: {len(ex['output'])} chars")

## 4. Prepare Dataset for Training

Convert Alpaca format to tokenized format, and set labels (only compute loss for the assistant's reply; set the prompt part to -100).

In [ ]:
MAX_SEQ_LENGTH = 4096

def format_training_example(item: dict) -> dict:
    """
    Convert a single Alpaca record to a tokenized format.
    - messages: [system, user, assistant] full conversation
    - Use apply_chat_template to get full text then tokenize
    - labels: set the prompt part to -100, only compute loss for the assistant's reply
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{item['instruction']}\n\nStatistics:\n{item['input']}"},
        {"role": "assistant", "content": item["output"]},
    ]
    # Full conversation (including assistant reply)
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Prompt only (excluding assistant reply), to calculate the length of the prompt
    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)
    prompt_len = prompt_ids.input_ids.shape[1]

    full_ids = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    input_ids = full_ids.input_ids[0]
    labels = input_ids.clone()
    labels[:prompt_len] = -100  # Do not compute loss for the prompt part

    return {"input_ids": input_ids, "labels": labels, "attention_mask": full_ids.attention_mask[0]}

# Convert all training samples
tokenized = [format_training_example(item) for item in train_data]
print(f"Tokenized {len(tokenized)} examples")
print(f"Example input_ids length: {len(tokenized[0]['input_ids'])}")

## 5. Create Dataset & Data Collator

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

# Convert to HuggingFace Dataset (tensors to list for Dataset compatibility)
def to_dataset_format(tok_list):
    return {
        "input_ids": [t["input_ids"].tolist() for t in tok_list],
        "labels": [t["labels"].tolist() for t in tok_list],
        "attention_mask": [t["attention_mask"].tolist() for t in tok_list],
    }

ds_dict = to_dataset_format(tokenized)
train_dataset = Dataset.from_dict(ds_dict)

# DataCollatorForSeq2Seq: Pads samples in the batch. Uses eos_token as pad_token_id
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100,
)

print(f"Dataset: {train_dataset}")

## 6. LoRA Config & PEFT Model

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# LoraConfig: r=rank, lora_alpha=scaling factor, target_modules are the layers where LoRA will be injected (SmolLM uses LLaMA architecture)
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Training Arguments & Trainer

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=str(adapter_path),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

## 8. Save Adapter

In [ ]:
trainer.save_model(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f"Saved adapter and tokenizer to: {adapter_path}")